In [ ]:
""" Convert timestamp strings into Spark TimestampType
→ calculate source-to-producer timing
→ validate required trade fields
→ display clean validation results """



from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    current_timestamp,
    from_json,
    lit,
    round,
    to_timestamp,
    when,
)
from pyspark.sql.types import (
    ArrayType,
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)


KAFKA_BOOTSTRAP_SERVERS = "YOUR_KAFKA_PRIVATE_IP:9092"
KAFKA_TOPIC = "stock-trades-raw"

CHECKPOINT_LOCATION = (
    "/home/ubuntu/stock-market-streaming/"
    "checkpoints/validated_console_test"
)

ALLOWED_SYMBOLS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "TSLA",
    "AMZN",
]


TRADE_SCHEMA = StructType(
    [
        StructField("schema_version", StringType(), True),
        StructField("event_id", StringType(), True),
        StructField("event_type", StringType(), True),
        StructField("source", StringType(), True),
        StructField("symbol", StringType(), True),
        StructField("price", DoubleType(), True),
        StructField("volume", DoubleType(), True),
        StructField(
            "trade_conditions",
            ArrayType(StringType()),
            True,
        ),
        StructField("event_timestamp_ms", LongType(), True),
        StructField("event_timestamp_utc", StringType(), True),
        StructField("ingestion_timestamp_utc", StringType(), True),
    ]
)


def main() -> None:
    spark = (
        SparkSession.builder
        .appName("StockMarketValidatedKafkaConsumer")
        .config("spark.sql.session.timeZone", "UTC")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    raw_stream = (
        spark.readStream
        .format("kafka")
        .option(
            "kafka.bootstrap.servers",
            KAFKA_BOOTSTRAP_SERVERS,
        )
        .option("subscribe", KAFKA_TOPIC)

        # Use latest for this test because the topic already
        # contains thousands of previous records.
        .option("startingOffsets", "latest")
        .load()
    )

    string_stream = raw_stream.selectExpr(
        "CAST(key AS STRING) AS message_key",
        "CAST(value AS STRING) AS message_value",
        "topic",
        "partition",
        "offset",
        "timestamp AS kafka_timestamp",
    )

    parsed_stream = (
        string_stream
        .withColumn(
            "trade",
            from_json(
                col("message_value"),
                TRADE_SCHEMA,
            ),
        )
        .select(
            col("trade.*"),
            col("message_key"),
            col("topic"),
            col("partition"),
            col("offset"),
            col("kafka_timestamp"),
        )
    )

    timestamped_stream = (
        parsed_stream
        .withColumn(
            "event_timestamp",
            to_timestamp(col("event_timestamp_utc")),
        )
        .withColumn(
            "ingestion_timestamp",
            to_timestamp(col("ingestion_timestamp_utc")),
        )
        .withColumn(
            "processing_timestamp",
            current_timestamp(),
        )
        .withColumn(
            "producer_latency_ms",
            round(
                (
                    col("ingestion_timestamp").cast("double")
                    - col("event_timestamp").cast("double")
                )
                * 1000,
                3,
            ),
        )
    )

    validated_stream = (
        timestamped_stream
        .withColumn(
            "validation_error",
            when(
                col("event_id").isNull(),
                "missing_event_id",
            )
            .when(
                col("symbol").isNull(),
                "missing_symbol",
            )
            .when(
                ~col("symbol").isin(ALLOWED_SYMBOLS),
                "unsupported_symbol",
            )
            .when(
                col("message_key") != col("symbol"),
                "key_symbol_mismatch",
            )
            .when(
                col("price").isNull() | (col("price") <= 0),
                "invalid_price",
            )
            .when(
                col("volume").isNull() | (col("volume") <= 0),
                "invalid_volume",
            )
            .when(
                col("event_timestamp").isNull(),
                "invalid_event_timestamp",
            )
            .when(
                col("ingestion_timestamp").isNull(),
                "invalid_ingestion_timestamp",
            )
            .otherwise(lit(None).cast("string")),
        )
        .withColumn(
            "is_valid",
            col("validation_error").isNull(),
        )
    )

    console_output = validated_stream.select(
        "symbol",
        "price",
        "volume",
        "event_timestamp",
        "ingestion_timestamp",
        "producer_latency_ms",
        "partition",
        "offset",
        "is_valid",
        "validation_error",
    )

    query = (
        console_output.writeStream
        .format("console")
        .outputMode("append")
        .option("truncate", "false")
        .option(
            "checkpointLocation",
            CHECKPOINT_LOCATION,
        )
        .trigger(processingTime="5 seconds")
        .start()
    )

    print("Validated Spark consumer is running.")
    print("Start the Finnhub producer to generate new records.")
    print("Press Ctrl+C to stop.")

    query.awaitTermination()


if __name__ == "__main__":
    main()